# Export MODIS Yield Proxies by Tile

Export annual yield proxy variables from three MODIS products, masked with annual cropland masks
(pre-computed at MODIS resolution from GLC-FCS30D 30m), aggregated to 10-km grid cells.

## Products
1. **MCD43A4** (NBAR 500m): annual max EVI, max GCVI, max harmonic-fitted GCVI - 1st priority export
2. **MOD13Q1** (VI 250m): annual max EVI - Optional export
3. **MOD13A1** (VI 500m): annual max EVI - 2nd priority export

All exports include annual cropland fraction and forest fraction.

## Prerequisites
Run `precompute_lc_fraction_modis.ipynb` first to create the 4 EE Image assets.


In [ ]:
import ee
import math

ee.Authenticate()
ee.Initialize()


In [ ]:
# Tiles and grid

farmSize = ee.Image(
    'users/hadicu06/Postdoc_Bonn/World_Bank_Project_2025/mode_farmsize_2010_reprojTo4326'
)
tiles = ee.FeatureCollection(
    'users/hadicu06/Postdoc_Bonn/World_Bank_Project_2025/'
    'export_tiles_v1_hasCroplandAndTreeLoss_30N_30S_Africa'
)


In [ ]:
# Parameters

ASSET_FOLDER = 'users/hadicu06/Postdoc_Bonn/World_Bank_Project_2025'

params = {
    'targetGrid': farmSize,
    'tiles': tiles,
    'start_date': '2001-01-01',
    'end_date': '2022-12-31',
    # Cropland masking at MODIS pixel level
    'modis_cropland_fraction_threshold': 0.0,  # any cropland (>0%) ## HH
    # Grid-level cropland selection
    'grid_cropland_fraction_threshold': 0.01,  # 1%
    'scale_compute_crop_fraction': 500,  # use 500m asset for grid selection
    # Harmonic fitting temporal resolution
    'harmonic_composite_days': 8,  # 1=daily (native), 8=8-day, 16=16-day  ## let's first try with 8
    # Pre-computed LC fraction assets (list = mosaic of tropics + north/south strips)
    'cropfrac_500m_asset': [
        f'{ASSET_FOLDER}/annual_cropfrac_modis_500m',        # 30S–30N
        f'{ASSET_FOLDER}/annual_cropfrac_modis_500m_north',  # 30–41N
        f'{ASSET_FOLDER}/annual_cropfrac_modis_500m_south',  # 36–30S
    ],
    # 'cropfrac_250m_asset': [f'{ASSET_FOLDER}/annual_cropfrac_modis_250m', ...],
    'forestfrac_500m_asset': [
        f'{ASSET_FOLDER}/annual_forestfrac_modis_500m',
        f'{ASSET_FOLDER}/annual_forestfrac_modis_500m_north',
        f'{ASSET_FOLDER}/annual_forestfrac_modis_500m_south',
    ],
    # 'forestfrac_250m_asset': [f'{ASSET_FOLDER}/annual_forestfrac_modis_250m', ...],
    # Export
    # 'version': '20260210', ## to change
    'version': '20260218', ## to change

    'gdrive': '',
    'debug': False,
}


In [ ]:
# ============================================================
# Helper functions
# ============================================================

def load_lc_fraction(asset_path):
    """Load pre-computed annual LC fraction asset (22 bands).

    Args:
        asset_path: str (single asset) or list of str (mosaicked assets,
            ordered so the first image takes priority where regions overlap).
            When a list is given, the mosaic's default projection is anchored
            to the first asset to prevent GEE falling back to 1° resolution.
    """
    if isinstance(asset_path, list):
        images = [ee.Image(p) for p in asset_path]
        mosaic = ee.ImageCollection(images).mosaic()
        # Use the first asset's full projection (CRS + native scale) so the
        # mosaic retains the correct MODIS resolution instead of 1°.
        ref_proj = images[0].projection()
        return mosaic.setDefaultProjection(crs=ref_proj)
    return ee.Image(asset_path)


def load_cropland_mask(asset_path, threshold=0.0):
    """Load cropland fraction asset and threshold to binary mask.

    Returns a 22-band binary image (1 where fraction > threshold).

    Args:
        asset_path: str (single asset) or list of str (mosaicked).
        threshold: pixels with cropfrac > threshold are included.
    """
    frac = load_lc_fraction(asset_path)
    return frac.gt(threshold)


def add_evi_modis(image):
    """Compute EVI from scaled MODIS reflectance (red, nir, blue bands)."""
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('nir'),
            'RED': image.select('red'),
            'BLUE': image.select('blue'),
        }
    ).rename('evi')
    return image.addBands(evi).copyProperties(
        image, ['system:time_start', 'system:time_end']
    )


def add_gcvi_modis(image):
    """Compute GCVI = NIR/GREEN - 1 from scaled MODIS reflectance."""
    gcvi = image.expression(
        'NIR / GREEN - 1',
        {
            'NIR': image.select('nir'),
            'GREEN': image.select('green'),
        }
    ).rename('gcvi')
    return image.addBands(gcvi).copyProperties(
        image, ['system:time_start', 'system:time_end']
    )


def annual_max_band(collection, year, band_name, output_prefix):
    """Compute annual max of a single band."""
    year = int(year)
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')
    filtered = collection.filterDate(start, end)
    return filtered.select(band_name).max().rename(f'{output_prefix}_{year}')


def harmonic_fit_and_max(gcvi_col, year, composite_days=1):
    """Fit 2nd-order harmonics on GCVI, return max of fitted curve.

    Args:
        gcvi_col: ImageCollection with 'gcvi' band (full date range, QA-filtered).
        year: Year to process.
        composite_days: 1=daily (native), 8=8-day, 16=16-day composites.
    """
    year = int(year)
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')
    yearly_gcvi = gcvi_col.filterDate(start, end)

    harmonic_names = ee.List(['constant', 'cos', 'sin', 'cos2', 'sin2'])

    if composite_days == 1:
        # Daily: derive DOY from system:time_start
        def _add_doy(image):
            doy = ee.Date(image.get('system:time_start')) \
                .getRelative('day', 'year').add(1)
            return image.addBands(
                ee.Image.constant(doy).rename('doy').float()
            )
        composites = yearly_gcvi.map(_add_doy)
    else:
        # N-day median composites
        n_periods = 365 // composite_days
        images = []
        for i in range(n_periods):
            p_start = start.advance(i * composite_days, 'day')
            p_end = p_start.advance(composite_days, 'day')
            doy_mid = i * composite_days + composite_days // 2 + 1
            filtered = yearly_gcvi.filterDate(p_start, p_end)
            composite = filtered.median().rename('gcvi')
            default = ee.Image.constant(0).rename('gcvi') \
                .updateMask(ee.Image.constant(0))
            result = ee.Image(
                ee.Algorithms.If(filtered.size().gt(0), composite, default)
            )
            result = result.addBands(
                ee.Image.constant(doy_mid).rename('doy').float()
            )
            images.append(result)
        composites = ee.ImageCollection.fromImages(images)

    # Add harmonic basis functions
    def _add_harmonics(image):
        doy = image.select('doy')
        t = doy.multiply(2 * math.pi).divide(365.25)
        return image \
            .addBands(ee.Image.constant(1).rename('constant')) \
            .addBands(t.cos().rename('cos')) \
            .addBands(t.sin().rename('sin')) \
            .addBands(t.multiply(2).cos().rename('cos2')) \
            .addBands(t.multiply(2).sin().rename('sin2'))

    harmonic_coll = composites.map(_add_harmonics)

    # Fit linear regression: harmonic terms -> GCVI
    coeffs = (
        harmonic_coll
        .select(harmonic_names.add('gcvi'))
        .reduce(ee.Reducer.linearRegression(
            numX=harmonic_names.length(), numY=1
        ))
        .select('coefficients')
        .arrayProject([0])
        .arrayFlatten([harmonic_names])
    )

    # Evaluate fitted curve every 8 days, take max
    eval_doys = ee.List.sequence(1, 365, 8)

    def _eval_at_doy(d):
        d = ee.Number(d)
        t = d.multiply(2 * math.pi).divide(365.25)
        basis = ee.Image.cat([
            ee.Image.constant(1),
            ee.Image.constant(t.cos()),
            ee.Image.constant(t.sin()),
            ee.Image.constant(t.multiply(2).cos()),
            ee.Image.constant(t.multiply(2).sin()),
        ]).rename(harmonic_names)
        return basis.multiply(coeffs).reduce(ee.Reducer.sum())

    fitted_vals = ee.ImageCollection(eval_doys.map(_eval_at_doy))
    return fitted_vals.max().rename(f'gcvi_harmonic_max_{year}')


def generate_grid(target_grid, aoi, cropfrac_asset_path,
                  grid_cropland_threshold, scale):
    """Generate 10-km grid filtered by cropland fraction threshold."""
    grid_image = target_grid
    proj = grid_image.projection()
    coords = ee.Image.pixelCoordinates(proj)
    unique_id = (
        coords.select('x').multiply(1000000)
        .add(coords.select('y')).toLong().rename('cell_id')
    )
    grid_image = grid_image.addBands([unique_id])

    grid_full = grid_image.select('cell_id').reduceToVectors(
        geometry=aoi,
        scale=10000,
        geometryType='polygon',
        labelProperty='cell_id',
    )

    # Use pre-computed cropland fraction (ever-cropland) for grid selection
    ever_cropfrac = load_lc_fraction(cropfrac_asset_path).reduce(ee.Reducer.max())
    crop_fraction = ever_cropfrac.unmask().reduceRegions(
        collection=grid_full,
        reducer=ee.Reducer.mean(),
        scale=scale,
    )
    grid = crop_fraction.filter(
        ee.Filter.gte('mean', grid_cropland_threshold)
    )

    # Add country and location metadata
    gaul = ee.FeatureCollection('FAO/GAUL/2015/level0')

    def _add_meta(f):
        centroid = f.geometry().centroid(1)
        lon = centroid.coordinates().get(0)
        lat = centroid.coordinates().get(1)
        gaul_f = gaul.filterBounds(centroid)
        country = ee.Algorithms.If(
            gaul_f.size().gt(0),
            gaul_f.first().get('ADM0_NAME'),
            ee.String('Unknown'),
        )
        return f.set({
            'country': country,
            'longitude': lon,
            'latitude': lat,
        })

    return grid.map(_add_meta)


In [ ]:
# ============================================================
# Product 1: MCD43A4 (NBAR 500m) - EVI, GCVI, harmonic GCVI
# ============================================================

def process_mcd43a4(aoi_buffered, grid, years, params,
                    filename_append, tile_id, debug=False):
    """Process MCD43A4 NBAR 500m: EVI max, GCVI max, GCVI harmonic max."""

    def filterBounds_(coll):
        return coll.filterBounds(aoi_buffered)

    def clip_(image):
        return image.clip(aoi_buffered)

    filter_time = ee.Filter.date(params['start_date'], params['end_date'])

    # --- Load NBAR reflectance ---
    nbar_col = (
        filterBounds_(ee.ImageCollection('MODIS/061/MCD43A4'))
        .filter(filter_time)
        .select(
            ['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band2',
             'Nadir_Reflectance_Band3', 'Nadir_Reflectance_Band4'],
            ['red', 'nir', 'blue', 'green']
        )
    )

    # --- Load QA ---
    qa_col = (
        filterBounds_(ee.ImageCollection('MODIS/061/MCD43A2'))
        .filter(filter_time)
        .select(
            ['BRDF_Albedo_Band_Quality_Band1',
             'BRDF_Albedo_Band_Quality_Band2',
             'BRDF_Albedo_Band_Quality_Band3',
             'BRDF_Albedo_Band_Quality_Band4',
             'BRDF_Albedo_Band_Quality_Band5',
             'BRDF_Albedo_Band_Quality_Band6',
             'BRDF_Albedo_Band_Quality_Band7'],
            ['qa1', 'qa2', 'qa3', 'qa4', 'qa5', 'qa6', 'qa7']
        )
    )

    # --- Join NBAR + QA by system:time_end ---
    join_filter = ee.Filter.equals(
        leftField='system:time_end', rightField='system:time_end'
    )
    inner_join = ee.Join.inner('NBAR', 'QA')
    joined = inner_join.apply(nbar_col, qa_col, join_filter)

    def _merge_nbar_qa(feature):
        nbar = ee.Image(feature.get('NBAR'))
        qa = ee.Image(feature.get('QA'))
        return nbar.addBands(qa)

    merged = ee.ImageCollection(joined.map(_merge_nbar_qa))

    # --- QA filter: all 7 bands <= 2, then scale reflectance ---
    # HH: to check, aha all ok
    # Including 2 (magnitude inversion, ≥7 observations) is a common compromise:
    # The example code in your prompt (and many MCD43A4 recipes) uses exactly this <=2 threshold, so this mirrors that practice.
    # If you prioritize denser time series for harmonic fitting over ultra-strict QA: the current <= 2 is reasonable.
    # BRDF_Albedo_Band_Quality_Band1 Bitmask	
    # Bits 0-2: BRDF inversion information for band 1
    # 0: Best quality, full inversion (WoDs and RMSE are good)
    # 1: Good quality, full inversion (also including the cases with no clear sky observations over the day of interest and those with a Solar Zenith Angle that is > 70 degrees even though WoDs and RMSE majority are good)
    # 2: Magnitude inversion (numobs >= 7)
    # 3: Magnitude inversion (numobs >= 2 & < 7)
    # 4: Fill value
    def _apply_qa_and_scale(image):
        qa_max = image.select(
            ['qa1', 'qa2', 'qa3', 'qa4', 'qa5', 'qa6', 'qa7']
        ).reduce(ee.Reducer.max())  ## to check if any band has qa value > 2 (i.e. not 0 or 1)
        qa_mask = qa_max.lte(2)
        scaled = image.select(
            ['red', 'nir', 'blue', 'green']
        ).multiply(0.0001)
        return scaled.updateMask(qa_mask).copyProperties(
            image, ['system:time_start', 'system:time_end']
        )

    filtered = merged.map(_apply_qa_and_scale)

    # --- Compute indices ---
    with_indices = filtered.map(add_evi_modis).map(add_gcvi_modis)
    evi_col = with_indices.select('evi')
    gcvi_col = with_indices.select('gcvi')

    # --- Annual max EVI, GCVI, harmonic GCVI ---
    evi_max_list = [
        annual_max_band(evi_col, y, 'evi', 'evi_max') for y in years
    ]
    gcvi_max_list = [
        annual_max_band(gcvi_col, y, 'gcvi', 'gcvi_max') for y in years
    ]
    gcvi_harm_list = [
        harmonic_fit_and_max(
            gcvi_col, y, params['harmonic_composite_days']
        )
        for y in years
    ]

    evi_max_bands = ee.Image.cat(evi_max_list)
    gcvi_max_bands = ee.Image.cat(gcvi_max_list)
    gcvi_harm_bands = ee.Image.cat(gcvi_harm_list)

    # --- Pre-computed LC fractions (500m) ---
    cropfrac = load_lc_fraction(params['cropfrac_500m_asset'])
    forestfrac = load_lc_fraction(params['forestfrac_500m_asset'])
    cropmask = load_cropland_mask(
        params['cropfrac_500m_asset'],
        params['modis_cropland_fraction_threshold'],
    )

    # --- Mask yield proxies with annual cropland mask ---
    evi_max_bands = clip_(evi_max_bands).updateMask(cropmask)
    gcvi_max_bands = clip_(gcvi_max_bands).updateMask(cropmask)
    gcvi_harm_bands = clip_(gcvi_harm_bands).updateMask(cropmask)

    # --- Stack: yield proxies (masked) + LC fractions (unmasked) ---
    cropfrac = clip_(cropfrac)
    forestfrac = clip_(forestfrac)
    stack = ee.Image.cat([
        evi_max_bands, gcvi_max_bands, gcvi_harm_bands,
        cropfrac, forestfrac,
    ])

    # --- ReduceRegions at 500m ---
    result = stack.reduceRegions(
        collection=grid,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4,
    )

    # Remove geometry for CSV export
    props = result.first().propertyNames().removeAll(['.geo'])
    result = result.select(props)

    # --- Export ---
    task = ee.batch.Export.table.toDrive(
        collection=result,
        description='grid_stack_mcd43a4' + filename_append,
        folder=params['gdrive'],
        fileFormat='CSV',
    )
    task.start()
    if debug:
        print(f'  MCD43A4 task started for tile {tile_id}')
    return task


In [ ]:
## Commented out, to check if updated
# # ============================================================
# # Product 2: MOD13Q1 (VI 250m) - EVI only
# # ============================================================

# def process_mod13q1(aoi_buffered, grid, years, params,
#                     filename_append, tile_id, debug=False):
#     """Process MOD13Q1 250m: annual max EVI."""

#     def filterBounds_(coll):
#         return coll.filterBounds(aoi_buffered)

#     def clip_(image):
#         return image.clip(aoi_buffered)

#     filter_time = ee.Filter.date(params['start_date'], params['end_date'])

#     # --- Load and QA-filter ---
#     def _mask_summary_qa(image):
#         qa = image.select('SummaryQA')
#         mask = qa.lte(1)  # 0=good, 1=marginal
#         evi = image.select('EVI').multiply(0.0001).rename('evi')
#         return evi.updateMask(mask).copyProperties(
#             image, ['system:time_start', 'system:time_end']
#         )

#     evi_col = (
#         filterBounds_(ee.ImageCollection('MODIS/061/MOD13Q1'))
#         .filter(filter_time)
#         .map(_mask_summary_qa)
#     )

#     # --- Annual max EVI ---
#     evi_max_list = [
#         annual_max_band(evi_col, y, 'evi', 'evi_max') for y in years
#     ]
#     evi_max_bands = ee.Image.cat(evi_max_list)

#     # --- Pre-computed LC fractions (250m) ---
#     cropfrac = load_lc_fraction(params['cropfrac_250m_asset'])
#     forestfrac = load_lc_fraction(params['forestfrac_250m_asset'])
#     cropmask = load_cropland_mask(
#         params['cropfrac_250m_asset'],
#         params['modis_cropland_fraction_threshold'],
#     )

#     # --- Mask and stack ---
#     evi_max_bands = clip_(evi_max_bands).updateMask(cropmask)
#     cropfrac = clip_(cropfrac)
#     forestfrac = clip_(forestfrac)
#     stack = ee.Image.cat([evi_max_bands, cropfrac, forestfrac])

#     # --- ReduceRegions at 250m ---
#     result = stack.reduceRegions(
#         collection=grid,
#         reducer=ee.Reducer.mean(),
#         scale=250,
#         tileScale=4,
#     )

#     props = result.first().propertyNames().removeAll(['.geo'])
#     result = result.select(props)

#     # --- Export ---
#     task = ee.batch.Export.table.toDrive(
#         collection=result,
#         description='grid_stack_mod13q1' + filename_append,
#         folder=params['gdrive'],
#         fileFormat='CSV',
#     )
#     task.start()
#     if debug:
#         print(f'  MOD13Q1 task started for tile {tile_id}')
#     return task


In [ ]:
# ============================================================
# Product 3: MOD13A1 (VI 500m) - EVI only
# ============================================================

def process_mod13a1(aoi_buffered, grid, years, params,
                    filename_append, tile_id, debug=False):
    """Process MOD13A1 500m: annual max EVI."""

    def filterBounds_(coll):
        return coll.filterBounds(aoi_buffered)

    def clip_(image):
        return image.clip(aoi_buffered)

    filter_time = ee.Filter.date(params['start_date'], params['end_date'])

    # --- Load and QA-filter ---
    def _mask_summary_qa(image):
        qa = image.select('SummaryQA')
        mask = qa.lte(1)  # 0=good, 1=marginal
        evi = image.select('EVI').multiply(0.0001).rename('evi')
        return evi.updateMask(mask).copyProperties(
            image, ['system:time_start', 'system:time_end']
        )

    evi_col = (
        filterBounds_(ee.ImageCollection('MODIS/061/MOD13A1'))
        .filter(filter_time)
        .map(_mask_summary_qa)
    )

    # --- Annual max EVI ---
    evi_max_list = [
        annual_max_band(evi_col, y, 'evi', 'evi_max') for y in years
    ]
    evi_max_bands = ee.Image.cat(evi_max_list)

    # --- Pre-computed LC fractions (500m) ---
    cropfrac = load_lc_fraction(params['cropfrac_500m_asset'])
    forestfrac = load_lc_fraction(params['forestfrac_500m_asset'])
    cropmask = load_cropland_mask(
        params['cropfrac_500m_asset'],
        params['modis_cropland_fraction_threshold'],
    )

    # --- Mask and stack ---
    evi_max_bands = clip_(evi_max_bands).updateMask(cropmask)
    cropfrac = clip_(cropfrac)
    forestfrac = clip_(forestfrac)
    stack = ee.Image.cat([evi_max_bands, cropfrac, forestfrac])

    # --- ReduceRegions at 500m ---
    result = stack.reduceRegions(
        collection=grid,
        reducer=ee.Reducer.mean(),
        scale=500,
        tileScale=4,
    )

    props = result.first().propertyNames().removeAll(['.geo'])
    result = result.select(props)

    # --- Export ---
    task = ee.batch.Export.table.toDrive(
        collection=result,
        description='grid_stack_mod13a1' + filename_append,
        folder=params['gdrive'],
        fileFormat='CSV',
    )
    task.start()
    if debug:
        print(f'  MOD13A1 task started for tile {tile_id}')
    return task


In [ ]:
# ============================================================
# Main tile processing function
# ============================================================

def process_export_tile(
    tile,
    filename_append,
    params,
    products=('mcd43a4', 'mod13a1'),
    debug=False,
):
    """Process and export MODIS products for one tile.

    Args:
        tile: ee.Feature with tileID property.
        filename_append: str suffix for export filenames.
        params: dict of pipeline parameters.
        products: tuple/list of product names to run. Any subset of:
            'mcd43a4' -- NBAR 500m: EVI max, GCVI max, harmonic GCVI max
            'mod13q1' -- VI 250m:  EVI max
            'mod13a1' -- VI 500m:  EVI max
        debug: bool, print extra info.
    """
    aoi = tile.geometry()
    tile_id = tile.get('tileID').getInfo()
    aoi_buffered = aoi.buffer(10000)

    # --- Generate shared 10-km grid ---
    grid = generate_grid(
        target_grid=params['targetGrid'],
        aoi=aoi,
        cropfrac_asset_path=params['cropfrac_500m_asset'],
        grid_cropland_threshold=params['grid_cropland_fraction_threshold'],
        scale=params['scale_compute_crop_fraction'],
    )

    if debug:
        print(f'Tile {tile_id}: grid size = {grid.size().getInfo()}')

    # --- Year range ---
    year_start = ee.Date(params['start_date']).get('year').getInfo()
    year_end = ee.Date(params['end_date']).get('year').getInfo()
    years = list(range(year_start, year_end + 1))

    # --- Process selected products ---
    selected = [p.lower() for p in products]
    n_started = 0

    if 'mcd43a4' in selected:
        process_mcd43a4(aoi_buffered, grid, years, params, filename_append, tile_id, debug)
        n_started += 1

    # if 'mod13q1' in selected:
    #     process_mod13q1(aoi_buffered, grid, years, params, filename_append, tile_id, debug)
    #     n_started += 1

    if 'mod13a1' in selected:
        process_mod13a1(aoi_buffered, grid, years, params, filename_append, tile_id, debug)
        n_started += 1

    print(f'Tile {tile_id}: {n_started} export task(s) started {selected}')



```
# Now you can control which products run by passing the products argument:

# Run only MCD43A4
process_export_tile(tile, filename, params, products=('mcd43a4',))

# Run only MOD13A1
process_export_tile(tile, filename, params, products=('mod13a1',))

# Run all three
process_export_tile(tile, filename, params, products=('mcd43a4', 'mod13q1', 'mod13a1'))

# Default (mcd43a4 + mod13a1, same as before)
process_export_tile(tile, filename, params)

# The default is ('mcd43a4', 'mod13a1') to preserve the previous behaviour.
```

In [ ]:
# Tile list
tile_ids_client = params['tiles'].aggregate_array('tileID').getInfo()
print(f'Total tiles: {len(tile_ids_client)}')


## Testing


In [ ]:
not_run = True

if not not_run: 

    # Test with a single tile
    test_tile_id = 63  

    test_tile = params['tiles'].filter(
        ee.Filter.eq('tileID', test_tile_id)
    ).first()

    process_export_tile(
        test_tile,
        f"__v_{params['version']}__t_{test_tile_id}__test",
        params,
        debug=True,
        # products=('mcd43a4', 'mod13a1')
        products=('mcd43a4',)
    )


## Full export runs


## Export for the daily MODIS product ('mcd43a4') first

In [ ]:
# Run ALL tiles
NOT_RUN = False  # Set to False to run

if not NOT_RUN:
    skip_ids = {621, 620, 615, 614, 294, 293, 285} ## checked before, these failed tiles are mostly ocean..
    for tile_id in tile_ids_client:
        if tile_id in skip_ids:
            print(f'Skipping tile: {tile_id}')
            continue
        tile = params['tiles'].filter(
            ee.Filter.eq('tileID', tile_id)
        ).first()
        process_export_tile(
            tile,
            f"__v_{params['version']}__t_{tile_id}",
            params,
            debug=False,
            products=('mcd43a4',)
        )

## Ran in laptop -> yes

In [ ]:
# # Run FIRST HALF of tiles
# NOT_RUN = True

# if not NOT_RUN:
#     skip_ids = {621, 620, 615, 614, 294, 293, 285}
#     half = len(tile_ids_client) // 2
#     for tile_id in tile_ids_client[:half]:
#         if tile_id in skip_ids:
#             print(f'Skipping tile: {tile_id}')
#             continue
#         tile = params['tiles'].filter(
#             ee.Filter.eq('tileID', tile_id)
#         ).first()
#         process_export_tile(
#             tile,
#             f"__v_{params['version']}__t_{tile_id}",
#             params,
#             debug=False,
#             products=('mcd43a4',)
#         )


# ## Ran in laptop


In [ ]:
# # Run SECOND HALF of tiles
# NOT_RUN = True

# if not NOT_RUN:
#     skip_ids = {621, 620, 615, 614, 294, 293, 285}
#     half = len(tile_ids_client) // 2
#     for tile_id in tile_ids_client[half:]:
#         if tile_id in skip_ids:
#             print(f'Skipping tile: {tile_id}')
#             continue
#         tile = params['tiles'].filter(
#             ee.Filter.eq('tileID', tile_id)
#         ).first()
#         process_export_tile(
#             tile,
#             f"__v_{params['version']}__t_{tile_id}",
#             params,
#             debug=False,
#             products=('mcd43a4',)
#         )


# ## Ran in laptop
